# Figures 5 and 6 -- persistence of the lead-day-1 eddies

Only the eddies present on forecast lead day 1 are kept, and each is followed for
as long as it is tracked. Figure 5 shows the surviving count and radius, Figure 6
the amplitude and speed.

The altimetry series is less a comparison than a control: it has no forecast, so
the same selection applied to observed eddies shows how much of the decline is
survivorship rather than forecast error. The file records that in its
`selection_caveat` attribute.


In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Folder holding the archived NetCDF files. Default: alongside this notebook.
DATA = Path(".")

FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

REGION_ORDER = ["Kuroshio", "Gulf of Mexico", "Arabian Sea"]


def by_region(paths):
    """Open each file, read its region attribute, return them in paper order."""
    tagged = []
    for p in paths:
        with xr.open_dataset(p) as ds:
            tagged.append((ds.attrs.get("region", p.stem), p))
    key = lambda t: (REGION_ORDER.index(t[0]) if t[0] in REGION_ORDER else 99, t[0])
    return sorted(tagged, key=key)


def need(pattern):
    found = sorted(DATA.glob(pattern))
    if not found:
        raise FileNotFoundError(
            f"No file matching {pattern!r} in {DATA.resolve()}. "
            f"Set DATA to the folder holding the archived NetCDF files.")
    return found


In [ ]:
COLOR = {"observed": "#2C3E50", "forecast": "#D81B60"}
STYLE = {"observed": dict(ls="-.", marker="D"),
         "forecast": dict(ls="-",  marker="o")}
LABEL = {"observed": "Altimetry", "forecast": "Forecast"}


def draw(ds, pair, ylabels, outname, suptitle=None):
    prods = [p for p in ("observed", "forecast") if p in ds["product"].values]
    rows = [(v, p) for v in pair for p in ds.polarity.values]
    nreg = ds.sizes["region"]
    fig, axes = plt.subplots(len(rows), nreg,
                             figsize=(7.5, 1.35 * len(rows) + 0.9),
                             dpi=130, sharex=True, squeeze=False)
    for i, (v, po) in enumerate(rows):
        for j, rg in enumerate(ds.region.values):
            ax = axes[i][j]
            for pr in prods:
                m  = ds[v].sel(region=rg, product=pr, polarity=po)
                se = ds[v + "_se"].sel(region=rg, product=pr, polarity=po)
                st = STYLE[str(pr)]
                ax.plot(ds.lead_day, m, st["ls"], marker=st["marker"], ms=2.5,
                        lw=1.4, color=COLOR[str(pr)], label=LABEL[str(pr)])
                ax.fill_between(ds.lead_day, m - se, m + se,
                                color=COLOR[str(pr)], alpha=0.18, lw=0)
            ax.grid(ls="--", lw=0.3)
            ax.tick_params(labelsize=7)
            ax.set_xticks(np.arange(1, 16, 2))
            ax.set_xlim(0.5, 15.5)
            ax.text(0.03, 0.94, f"({chr(97 + i * nreg + j)})",
                    transform=ax.transAxes, fontsize=7.5, fontweight="bold",
                    va="top")
            if i == 0:
                ax.set_title(str(rg), fontsize=8.5, fontweight="bold")
            if j == 0:
                ax.set_ylabel(f"{ylabels[v]}\n{str(po).capitalize()}", fontsize=7)
            if i == len(rows) - 1:
                ax.set_xlabel("Forecast lead day", fontsize=7.5)
    h, l = axes[0][0].get_legend_handles_labels()
    fig.legend(h[:len(prods)], l[:len(prods)], loc="lower center",
               ncol=len(prods), fontsize=7.5, frameon=False,
               bbox_to_anchor=(0.5, -0.015))
    if suptitle:
        fig.suptitle(suptitle, fontsize=9)
    fig.tight_layout(h_pad=0.4, w_pad=0.5, rect=(0, 0.03, 1, 1))
    out = FIGDIR / outname
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("wrote", out)


In [ ]:
path = need("Figures5_6_*.nc")[0]
with xr.open_dataset(path) as ds:
    ylab = {"n_surviving": "day-1 eddies",
            "radius":      "Radius (km)",
            "amplitude":   "Amplitude (cm)",
            "speed":       "Speed (cm s$^{-1}$)"}
    sup = "Eddies present on lead day 1, followed while they live"
    draw(ds, ["n_surviving", "radius"], ylab,
         "Figure5_number_radius.png", suptitle=sup)
    draw(ds, ["amplitude", "speed"], ylab,
         "Figure6_amplitude_speed.png", suptitle=sup)

    print("\nsurvival fraction, day 15 / day 1")
    print(f"{'region':16s}{'product':10s}{'polarity':14s}{'fraction':>10s}")
    for rg in ds.region.values:
        for pr in ds["product"].values:
            for po in ds.polarity.values:
                s = ds.n_surviving.sel(region=rg, product=pr, polarity=po)
                n1, n15 = float(s.sel(lead_day=1)), float(s.sel(lead_day=15))
                print(f"{str(rg):16s}{str(pr):10s}{str(po):14s}"
                      f"{100 * n15 / n1:9.1f}%")
    if "licence" in ds.attrs:
        print("\n" + ds.attrs["licence"])
